In [ ]:
import redis, os, sys, time, threading
import pandas as pd
import random
from dfconnect import *
from time import sleep
from datetime import datetime
from df_workload_gen import main as wl_main

# connection info often passed on the command line:

# local Redis container:
sys.argv.extend(['--host', '127.0.0.1', '--port', '10000', 'socket_timeout', '25', 'socket_connect_timeout', '15'])


# add args for df_workload_gen
# "--duration" "--client-name"
sys.argv.extend(['--duration', '30', '--client-name', 'ot-workbook'])

df = connect_to_datastore()

def dfo(*args):
    s = None
    try:
        s = df.execute_command(*args)
    except Exception as e:
        s = f'Issue with command: {e}'
        print(s)
    return s
        

#print(df)

print(f'DBSIZE == {dfo("dbsize")}')


In [ ]:
# STRING EXAMPLES:
print(dfo("SET","myString","Hi Owen"))
print(dfo("APPEND","myString"," ... your name is Owen... right?"))
print(dfo("GET","myString"))
dfo("del","ot_count")
print(dfo("INCRBY","ot_count","6"))
print(dfo("INCRBY","ot_count","2"))
print(dfo("INCRBYFLOAT","ot_count","1.6"))
print(dfo("INCRBYFLOAT","ot_count","-7.1"))

In [ ]:
# RATE LIMITING EXAMPLE:
loop=0
loop2=0

while loop < 6:
    loop=loop+1
    sleep(.25)
    print(dfo("CL.THROTTLE", "ratelimit:AsciiArtServer", "2", "3", "10", "1"))

# CL.THROTTLE <key> <max_burst> <count per period> <period> [<quantity-used>]

while loop2 < 5:
    loop2=loop2+1
    sleep(2)
    print(f'ratelimit key in use is:  {dfo("keys","ratelimit:AsciiArtServer")}')

# 1,  ← if this response is 0 then allow access.  (If it is 1 then disallow access)
# 3,  ← what is maximum capacity for this THROTTLE
# 0,  ← What is remaining number of tokens available
# 3,  ← how many seconds remain before any access is possible
# 10  ← how many seconds remain before THROTTLE is fully RESET


In [ ]:
# user preferences / Recommendation Example
dfo("del","user:1:offered")
dfo("del","user:1:deselected")
dfo("SADD","user:1:offered","1","2","3","4","5","6")
dfo("SADD","user:1:deselected","1","2","3","5")
print(f'remaining suggestions {dfo("SDIFF","user:1:offered","user:1:deselected")}')

In [ ]:
# DEDUPLICATION (prevent outbound duplicate messages or inbound duplicate processing)
loop=0
while loop<10:
    loop=loop+1
    res=dfo("BF.ADD", "cust_id", "12345:"+str(loop))
    if (res!=1):
        print(f"DUPLICATE ENTRY! --> {"12345:"+str(loop)}")
print(f'cust_id key has been set to expire in 10 seconds? {dfo("EXPIRE","cust_id","10")}')

#use SHIFT|ENTER to display image in this cell (runs the cell)
![IMG of_flow](jobWorkflow1.png)

In [ ]:
# MANAGE A JOB QUEUE USING blmove with LISTS:
import itertools
# Cycles back and forth between 0 and 1 indefinitely
router_cycle = itertools.cycle([0, 1])

def process_next_job(df,worker):
    router = next(router_cycle)
    # 1. SAFELY READ from the queue by moving it to the processing list.
    # This keeps the item safe if the script crashes right after this line.
    job_payload = df.blmove("L:main:{"+str(router)+"}", "L:processing:{"+str(router)+"}", src="RIGHT", dest="LEFT", timeout=2)
    print(f"job_payload =={job_payload}")
    
    # If the queue was empty and timed out, exit early cleanly
    if not job_payload:
        return "\n\t⚠️⚠️⚠️ NO MORE JOBS ⚠️⚠️⚠️"
        
    try:
        # 2. CONCATENATE the payload string to target your corresponding hash key safely
        # Example: if router is 0 and job_payload is "1", hash_key becomes "h:job_details:{0}:1"
        hash_key = "h:job_details:{"+str(router)+"}:"+str(job_payload)
        
        # 3. READ the corresponding detail fields from the Hash data store
        # Use hgetall to pull all task configuration properties securely
        job_details = df.hgetall(hash_key)
        
        if not job_details:
            # Handle Edge Case: The queue element exists, but the hash data was expired/missing
            print(f"⚠️ Warning: Found queue item {job_payload} but data store {hash_key} is empty!")
            # Move the orphaned queue item out of processing so it doesn't get stuck
            df.lrem("L:processing:{"+str(router)+"}", 1, job_payload)
            return "⚠️ ORPHANED_JOB_CLEANED"
            
        # 4. RUN YOUR BUSINESS LOGIC HERE using the hash data
        print(f"{worker} Processing task : {job_details.get('AISLE')}")        
        df.hset(hash_key,'AISLE','COMPLETE')
        
        # 5. ACKNOWLEDGE COMPLETION: Only delete from the processing safety net 
        # AFTER your business logic successfully runs without throwing exceptions.
        df.lrem("L:processing:{"+str(router)+"}", 1, job_payload)
        
        # (Optional) Clean up or expire the hash data now that the job is completely done
        df.delete(hash_key)
        
        return f"\n✅ {worker} SUCCESS {datetime.now().strftime("%H:%M:%S:%s")} PROCESSED:{job_payload} \nJOB_DETAILS:{job_details}"
        
    except Exception as e:
        # If anything crashed during your business logic or hash lookup, 
        # do NOT run LREM. The item stays safe in 'L:processing' for recovery scripts.
        print(f"❌ {worker} Critical error while processing {job_payload}: {e}")
        raise e


def background_task(name):
    loop_count = 8
    for i in range(loop_count):
        print(f"[{name}] working: loop {i+1} of {loop_count} \n")
        print(process_next_job(df,name))

# Spawn two distinct "process_next_job" threads:
t1 = threading.Thread(target=background_task, args=("Worker-A",))
t2 = threading.Thread(target=background_task, args=("Worker-B",))

t1.start()
t2.start()

# Back to main thread to populate job queue: (one extra job on purpose)
loop=0

# create an orphaned job with no details:
dfo("LPUSH","L:main:{0}","no_matching_job_payload(on_purpose):{0}")

#publish workable jobs:
while loop<9:
    loop = loop+1
    router=loop%2
    dfo("HSET", "h:job_details:{"+str(router)+"}:"+str(loop), "TASK", "Cleanup on aisle", "AISLE", str(loop))
    dfo("LPUSH","L:main:{"+str(router)+"}",str(loop))
    
dfo("EXPIRE","L:main:{0}","120")
dfo("EXPIRE","L:processing:{0}","120")
dfo("EXPIRE","L:main:{1}","120")
dfo("EXPIRE","L:processing:{1}","120")

In [ ]:
# STREAMS Job Queue EXAMPLE:
# MANAGE A JOB QUEUE with some workers: USING STREAMS
# Migrating from a Redis List queue (BLMOVE) to Redis Streams with Consumer Groups (XREADGROUP) is a major upgrade.

# Instead of moving items between different lists to ensure safety, Redis Streams tracks delivery state natively inside the stream using a Pending Entries List (PEL). When a worker reads a message, Redis automatically tracks that the message is "in-flight" for that specific worker. Once processed successfully, the worker acknowledges it (XACK), removing it from the PEL.

# Here is the complete, redesigned script utilizing Redis Streams, consumer groups, and proper error handling.
# Configuration Constants
STREAM_NAME = "S:jobs_stream"
GROUP_NAME = "G:worker_group"

def initialize_stream_and_group():
    """Ensures the stream and consumer group exist before workers start."""
    dfo("del","S:jobs_stream")
    
    try:
        # Create consumer group pointing to '$' (only receive new messages moving forward)
       # mkstream=True automatically initializes the stream if it doesn't exist
        df.xgroup_create(STREAM_NAME, GROUP_NAME, id="$", mkstream=True)
        print(f"✅ Consumer Group '{GROUP_NAME}' created successfully.")
    except redis.exceptions.ResponseError as e:
        if "BUSYGROUP" in str(e):
            print(f"ℹ️ Consumer Group '{GROUP_NAME}' already exists. Skipping setup. error looks like: {e}")
        else:
            raise e


def process_next_job(df, worker_name):
    # 1. SAFELY READ from the stream using the Consumer Group.
    # '>' means: read messages that have never been delivered to any other consumer.
    try:
        response = df.xreadgroup(
            groupname=GROUP_NAME,
            consumername=worker_name,
            streams={STREAM_NAME: ">"},
            count=1,
            block=2000,  # Timeout in milliseconds (2 seconds)
        )
    except redis.exceptions.RedisError as e:
        print(f"❌ {worker_name} Stream read error: {e}")
        return None

    # If the stream was empty and timed out, exit early cleanly
    if not response:
        return f"\n\t⚠️⚠️⚠️ {worker_name}: NO MORE NEW JOBS ⚠️⚠️⚠️"

    # Parse stream payload structures: [[stream_name, [[message_id, {data}]]]]
    _, messages = response[0]
    message_id, data = messages[0]
    job_payload = data.get("payload_id")

    try:
        # 2. TARGET corresponding hash key safely
        hash_key = f"h:job_details:{job_payload}"

        # 3. READ detail fields from the Hash data store
        job_details = df.hgetall(hash_key)

        if not job_details:
            # Handle Edge Case: The stream entry exists, but the hash data was expired/missing
            print(
                f"⚠️ Warning: Found queue item {job_payload} but data store {hash_key} is empty!"
            )
            # Acknowledge the message anyway so it is removed from the safety net (PEL)
            df.xack(STREAM_NAME, GROUP_NAME, message_id)
            # Optionally delete the broken stream entry completely
            df.xdel(STREAM_NAME, message_id)
            return f"⚠️ {worker_name}: ORPHANED_JOB_CLEANED"

        # 4. RUN YOUR BUSINESS LOGIC HERE using the hash data
        print(f"[{worker_name}] Processing task : {job_details.get('AISLE')}")
        time.sleep(0.01)
        df.hset(hash_key, "AISLE", "COMPLETE")

        # 5. ACKNOWLEDGE COMPLETION: This removes the item from this worker's
        # Pending Entries List (PEL) safety net.
        df.xack(STREAM_NAME, GROUP_NAME, message_id)

        # Clean up or expire the hash data now that the job is completely done
        df.delete(hash_key)
        # Clean up the stream entry to save memory
        df.xdel(STREAM_NAME, message_id)

        timestamp = datetime.now().strftime("%H:%M:%S.%s")
        return f"✅ {worker_name} SUCCESS {timestamp} PROCESSED:{job_payload} \nJOB_DETAILS:{job_details}"

    except Exception as e:
        # If anything crashed during your business logic, do NOT run XACK.
        # The item stays safe inside the Consumer Group PEL for a recovery/dead-letter script.
        print(
            f"❌ {worker_name} Critical error while processing ID {message_id} (Payload: {job_payload}): {e}"
        )
        raise e


def background_task(name):
    for i in range(9):
        result = process_next_job(df, name)
        if result:
            print(result)
        time.sleep(1.5)


# --- Execution Execution Orchestration ---

# 1. Setup Stream Environment Natively
initialize_stream_and_group()

# 2. Spawn two distinct worker threads mapped to the consumer group
t1 = threading.Thread(target=background_task, args=("Worker-A",))
t2 = threading.Thread(target=background_task, args=("Worker-B",))

# starting the Stream workers to begin processing events of the stream:
t1.start()
time.sleep(0.1)
t2.start()

# 3. Main thread populates job stream
loop = 0

# Create an orphaned job with no details in the stream:
df.xadd(
    STREAM_NAME, {"payload_id": f"no_matching_job_payload(on_purpose):{loop}"}
)

print(f'STREAM EXISTS? --> {dfo("EXISTS",STREAM_NAME)}')
print(f'STREAM EXISTS? --> {dfo("EXISTS",STREAM_NAME)}')

# Publish workable jobs using XADD
while loop < 11:
    loop += 1
    # Create details hash
    df.hset(
        f"h:job_details:{loop}",
        mapping={"TASK": "Cleanup on aisle", "AISLE": str(loop)},
    )
    # Add payload entry reference to stream ('*' generates automatic chronological ID)
    df.xadd(STREAM_NAME, {"payload_id": str(loop)})
    time.sleep(0.25)

print(f'\n\nMAIN THREAD... XINFO: {dfo("XINFO CONSUMERS", "S:jobs_stream", "G:worker_group")}\n\n')


In [ ]:
# generate load test all types throughput with parallel threads:
wl_main()

print(f'DBSIZE == {dfo("dbsize")}')


In [ ]:
# USE PANDAS to pretty print info regarding dragonfly:
dataframe = pd.DataFrame(dfo("INFO","commandstats")).T
dataframe.index = dataframe.index.str.replace("cmdstat_", "")
dataframe.index.name = "Command"
# Sort by total calls descending
dataframe = dataframe.sort_values(by="calls", ascending=False)
print(dataframe)

In [ ]:
# Combine Search with Streaming data:

## To enable orders to be placed on a stream
# create sets of attributes to be selected for the orders:
# only needs to be executed one time:
dfo("EVAL", "do redis.call('SADD',KEYS[1],'10.00:3d movie ticket') redis.call('SADD',KEYS[1],'2.00:bottle of water') redis.call('SADD',KEYS[1],'3.00:bottle of juice') redis.call('SADD',KEYS[1],'8.00:Turkey Burger') redis.call('SADD',KEYS[1],'5.00:French Fries') redis.call('SADD',KEYS[1],'6.00:Fruit Smoothie') redis.call('SADD',KEYS[1],'8.00:Vegetable Wrap') redis.call('SADD',KEYS[1],'10.00:book') redis.call('SADD',KEYS[1],'15.00:stuffed animal') redis.call('SADD',KEYS[1],'1.00:pin') redis.call('SADD',KEYS[1],'2.00:postcard') redis.call('SADD',KEYS[1],'5.00:child entrance ticket') redis.call('SADD',KEYS[1],'15.00:Adult entrance ticket') redis.call('SADD',KEYS[1],'1.50:lollipop') redis.call('SADD',KEYS[1],'12.00:jigsaw puzzle') end", "1", "zew:{batch2}:common")
dfo("EVAL", "do  redis.call('SADD',KEYS[1],'8.00:Senior Entrance Ticket') redis.call('SADD',KEYS[1],'80.00:Family Annual Subscription') redis.call('SADD',KEYS[1],'16.00:Butterfly Hall Experience') redis.call('SADD',KEYS[1],'60.00:Hardcover Book') end", "1", "zew:{batch2}:rare")

## To generate the Search Index: 
try:
    dfo("FT.DROPINDEX","idx_zew_revenue")
except Exception as e:
    print(f'FT.DROPINDEX FAILED with {e}')  

dfo("FT.CREATE","idx_zew_revenue", \
    "PREFIX","1","zew:revenue:", \
    "SCHEMA","visitor_purchase_item_name","TAG", \
    "visitor_purchase_item_cost","NUMERIC","SORTABLE")



In [ ]:
# create stream workergroup to process the events:
try:
    df.xgroup_create('zew:{batch2}:revenue:stream','group1','0-0','MKSTREAM')
except Exception as e:
    print(f'XGROUP_CREATE ... group already exists .. continuing on...{e}')  


In [ ]:
#Test lua looking at keys:
print(f'{dfo("EVAL","local outs = 'key1: '..redis.call('EXISTS', KEYS[1])..' key2: '..redis.call('EXISTS', KEYS[2])..' key3: '..redis.call('EXISTS', KEYS[3]) return outs", "3", "zew:{batch2}:revenue:stream","zew:{batch2}:common","zew:{batch2}:rare")}')
print(f'{df.execute_command("EVAL","local outs = 'key1: '..redis.call('EXISTS', KEYS[1])..' key2: '..redis.call('EXISTS', KEYS[2])..' key3: '..redis.call('EXISTS', KEYS[3]) return outs", "3", "zew:{batch2}:revenue:stream","zew:{batch2}:common","zew:{batch2}:rare")}')


In [ ]:
# designed to run repeatedly in order to populate many orders: 100*50 == 5000
ordercnt=0
while ordercnt<100:
    #dfo("EVAL", "for index = 1,50 do redis.call('XADD', KEYS[1],'*','visitor_purchase', index %3 == 0 and redis.call('SRANDMEMBER', KEYS[3]) or redis.call('SRANDMEMBER', KEYS[2])) end return 'Added 50 park visitor purchase events'", "3", "zew:{batch2}:revenue:stream", "zew:{batch2}:common", "zew:{batch2}:rare")
    dfo("EVAL", """
    -- 1. Check if the sets exist and have elements
    local stream_exists = redis.call('EXISTS', KEYS[1])
    local common_exists = redis.call('EXISTS', KEYS[2])
    local rare_exists = redis.call('EXISTS', KEYS[3])

    -- 2. Return explicit error messages if either is missing
    if stream_exists == 0 then
        return redis.error_reply('Key not found or empty: ' .. KEYS[1])
    end
    if rare_exists == 0 then
        return redis.error_reply('Key not found or empty: ' .. KEYS[3])
    end
    if common_exists == 0 then
        return redis.error_reply('Key not found or empty: ' .. KEYS[2])
    end
    for index = 1,50 do
        local member
        if index % 3 == 0 then
            member = redis.call('SRANDMEMBER', KEYS[3])
        else
            member = redis.call('SRANDMEMBER', KEYS[2])
        end
        
        -- Only add to the stream if we actually found a member
        if member then
            redis.call('XADD', KEYS[1], '*', 'visitor_purchase', member)
        end
    end 
    return 'Added 50 park visitor purchase events'
    """, "3", "zew:{batch2}:revenue:stream", "zew:{batch2}:common", "zew:{batch2}:rare")    
    ordercnt=ordercnt+1
    sleep(.1) # execute 10 loops / second

In [ ]:
# run 5 threads that each process 100 events (total 500 events written as hashes)

def convert_events(worker_name):
    # now, to process order events and convert them into searchable objects:
    numberofevents = 100
    # Have a single worker belonging to our group process 500 stream events
    # using the > character tells redis to only deliver events unprocessed by this group: 
    streamsdict = {'zew:{batch2}:revenue:stream': ">"}
    for x in range(numberofevents):
        try:
            response = df.xreadgroup('group1',worker_name,streams=streamsdict,count=1,noack=False)
            eventid = response[0][1][0][0] # the id assigned to the event when it was created
            astring = response[0][1][0][1].get('visitor_purchase') # compound string (attribute of the event)
            itemcost = astring.split(":").pop(0) # by programmer choice the cost and name are stored together
            itemname = astring.split(":").pop(1) # by programmer choice the cost and name are stored together
            # create a Hash record for the processed event:
            df.hset('zew:revenue:txid'+eventid,mapping={'visitor_purchase_item_name':itemname,'visitor_purchase_item_cost':itemcost})
            if(x%100==0):
                print(f"{worker_name} sample {x} of records written: {df.hgetall('zew:revenue:txid'+eventid)}")
            if(x >= (numberofevents-1)): #zero indexing expected for x
                print(f"{worker_name} sample {numberofevents} of records written: {df.hgetall('zew:revenue:txid'+eventid)}")
            df.xack('zew:{batch2}:revenue:stream','group1',eventid)
        except:
            if((x%100==0) or (x >= (numberofevents-1))):
                print('There are no more items in this stream to be processed by this group')
                length = df.execute_command('XLEN',"zew:{batch2}:revenue:stream")
                print(f"There are {length} items in the stream")

# 2. Spawn 5 distinct worker threads mapped to the consumer group
t1 = threading.Thread(target=convert_events, args=("Worker-A",))
t2 = threading.Thread(target=convert_events, args=("Worker-B",))
t3 = threading.Thread(target=convert_events, args=("Worker-C",))
t4 = threading.Thread(target=convert_events, args=("Worker-D",))
t5 = threading.Thread(target=convert_events, args=("Worker-E",))

# starting the Stream workers to begin processing events of the stream:
t1.start()
t2.start()
time.sleep(0.1)
t3.start()
t4.start()
time.sleep(0.1)
t5.start()

In [ ]:
# Now query the objects using FT.SEARCH:
# use redis search to query the set of indexed Hashes:
sresult = df.execute_command("FT.AGGREGATE","idx_zew_revenue", "@visitor_purchase_item_cost:[1 80]", "GROUPBY", "1", \
                             "@visitor_purchase_item_name", "reduce", "SUM", "1", "@visitor_purchase_item_cost", \
                             "AS", "total_earned", "GROUPBY", "2", \
                             "@visitor_purchase_item_name", "@total_earned", \
                             "SORTBY", "2", "@total_earned", "DESC", "LIMIT", "0", "100", "DIALECT","3")

print(sresult)
try:
    for c in range(len(sresult)):
        print(sresult[c])
except Exception as e:
    print('Different format result from search response')

    # 2. Extract the data nested inside 'extra_attributes'
    data_list = [item['extra_attributes'] for item in sresult['results']]

    # 3. Create the DataFrame and convert numeric strings to integers
    dataframe = pd.DataFrame(data_list)
    dataframe['total_earned'] = pd.to_numeric(dataframe['total_earned'])
    
    # 4. Set the index to the item name and rename the index label
    dataframe.set_index('visitor_purchase_item_name', inplace=True)
    dataframe.index.name = "Item Name"
    
    # 5. Sort by total earned descending
    dataframe = dataframe.sort_values(by="total_earned", ascending=False)
    
    # 6. Apply US dollar formatting for display/printing
    dataframe['total_earned'] = dataframe['total_earned'].map('${:,.2f}'.format)
    
    print(dataframe)
    




qstring = ''' "FT.AGGREGATE" "idx_zew_revenue"
          "@visitor_purchase_item_cost:[1 80]"
          "GROUPBY" "1" "@visitor_purchase_item_name"
          "reduce" "SUM" "1" "@visitor_purchase_item_cost"
          "AS" "total_earned" "GROUPBY" "2" "@visitor_purchase_item_name" "@total_earned"
          "SORTBY" "2" "@total_earned" "DESC"
          "LIMIT" "0" "100" "DIALECT" "3" '''

print(f"\n\nAbove results came from this query: {qstring}")

In [ ]:
# Use python Script to check compatibility
from check_dragonfly_compat import *

dataframe = build_dataframe(df)
# Print summary table
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 40)

display_cols = ["calls", "usec_per_call", "dragonfly_support", "cluster_mode"]
available = [c for c in display_cols if c in dataframe.columns]
print(dataframe[available].to_string())

# Summary counts
print("\n--- Summary ---")
counts = dataframe["dragonfly_support"].value_counts()
for status, n in counts.items():
    print(f"  {status}: {n} command(s)")

cluster_issues = dataframe[dataframe["cluster_mode"] != ""]
if not cluster_issues.empty:
    print(f"\n  Cluster-mode restricted: {len(cluster_issues)} command(s)")

unsupported = dataframe[dataframe["dragonfly_support"].isin(["Unsupported", "Partial"])]
if not unsupported.empty:
    print("\n--- Commands needing attention ---")
    print(unsupported[available].to_string())

if not cluster_issues.empty:
    print("\n--- Cluster mode restrictions ---")
    print("  Disabled   = command errors in cluster mode regardless of arguments")
    print("  Cross-slot = command errors if keys span different hash slots; use {hash-tag} to co-locate")
    print()
    print(cluster_issues[available].to_string())



In [ ]:
# lua script that converts JSON objects into binary string keys (for ease of transfer to a new datastore)
lua_to_string = '''local cursor = "0"
local count = 0

repeat
    -- Scan for keys (you can add a MATCH pattern here if needed, e.g., 'user:*')
    local result = redis.call('SCAN', cursor, 'MATCH', '*', 'COUNT', 100, 'TYPE','ReJSON-RL')
    cursor = result[1]
    local keys = result[2]

    for _, keyname in ipairs(keys) do
        -- Check the type to ensure it's a RedisJSON key
        local keytype = redis.call('TYPE', keyname)['ok']
        
        if keytype == 'ReJSON-RL' then
            -- Get the JSON string (using '$' returns an array of the match)
            local jsonVal = redis.call('JSON.GET', keyname, '$')
            
            if jsonVal then
                -- Store it as a plain string with your prefix
                redis.call('SET', 'jsonTempString:' .. keyname, jsonVal)
                count = count + 1
            end
        end
    end

until cursor == "0"

return "Successfully converted " .. count .. " JSON keys to strings."'''
#NB: I fully expect this to fail due to slot assignment issues...
print(f'Result from lua_to_stringa: {dfo("EVAL",lua_to_string,'0')}')

In [ ]:
# convert binary strings into JSON (reversing the above)
# this allows for tools like redisShake to manage the replication of JSON keys as temporary Strings

lua_to_json = '''local cursor = "0"
local prefix = "jsonTempString:"
local prefix_len = string.len(prefix)
local count = 0

repeat
    -- Scan specifically for keys matching your prefix
    local result = redis.call('SCAN', cursor, 'MATCH', prefix .. '*', 'COUNT', 100)
    cursor = result[1]
    local keys = result[2]

    for _, tempKey in ipairs(keys) do
        -- Read the stored JSON string value
        local jsonStr = redis.call('GET', tempKey)
        
        if jsonStr then
            -- Extract original key name (everything after 'jsonTempString:')
            local originalKey = string.sub(tempKey, prefix_len + 1)
            
            -- If the string has the outer array wrapper '[...]' from the '$' GET, 
            -- we strip the first and last character so it becomes a raw object again.
            if string.sub(jsonStr, 1, 1) == "[" and string.sub(jsonStr, -1, -1) == "]" then
                jsonStr = string.sub(jsonStr, 2, -2)
            end            
            
            -- Restore as a RedisJSON object
            -- Note: '$' expects a valid root element. If your string is wrapped in 
            -- an array (e.g., '[{...}]'), '$' will restore it as an array.
            redis.call('JSON.SET', originalKey, '.', jsonStr)
            redis.call('DEL', tempKey)
            
            -- ^ Optional: Delete the temporary string key to clean up space
            
            count = count + 1
        end
    end

until cursor == "0"

return "Successfully restored " .. count .. " keys back to RedisJSON."'''
#NB: I fully expect this to fail due to slot assignment issues...
print(f'Result from lua_to_json: {dfo("EVAL",lua_to_json,'0')}')

In [ ]:
# lua script that converts topk keys into ZSet keys (for ease of transfer to a new datastore)
lua_to_zset='''local cursor = "0"
local count = 0

-- Helper: unwrap a Redis reply that may be a table (bulk string) or a raw value
local function unwrap(v)
    if type(v) == 'table' then
        return v[1] or v['ok'] or v['err'] or ''
    end
    return tostring(v)
end

repeat
    local result = redis.call('SCAN', cursor, 'MATCH', 'topk:*', 'COUNT', 100, 'TYPE', 'TOPK-TYPE')
    cursor = result[1]
    local keys = result[2]

    for _, keyname in ipairs(keys) do
        local info = redis.call('TOPK.INFO', keyname)

        -- Walk info pairs to find depth
        local depth = 0
        local i = 1
        while i <= #info - 1 do
            local field = unwrap(info[i])
            local val   = unwrap(info[i + 1])
            if field == 'depth' then
                depth = tonumber(val) or 0
                break
            end
            i = i + 2
        end

        if depth > 0 then
            local list   = redis.call('TOPK.LIST', keyname, 'WITHCOUNT')
            local zsetKey = 'topkTempZSet:' .. keyname

            local j = 1
            while j <= #list - 1 do
                local item  = unwrap(list[j])
                local score = tonumber(unwrap(list[j + 1])) or 0
                redis.call('ZADD', zsetKey, score, item)
                j = j + 2
            end

            count = count + 1
        end
    end

until cursor == "0"

return "Successfully exported " .. count .. " Top-K keys to sorted sets."'''
print(f'Result from lua_to_json: {dfo("EVAL",lua_to_zset,'0')}')

In [ ]:
# convert ZSet objects into TOPK keys (opposite of above)
lua_to_topk='''local cursor = "0"
local count = 0
local skipped = {}

-- Helper: unwrap a Redis reply that may be a table (bulk string) or a raw value
local function unwrap(v)
    if type(v) == 'table' then
        return v[1] or v['ok'] or v['err'] or ''
    end
    return tostring(v)
end

repeat
    local result = redis.call('SCAN', cursor, 'MATCH', 'topkTempZSet:*', 'COUNT', 100, 'TYPE', 'zset')
    cursor = result[1]
    local keys = result[2]

    for _, zsetKey in ipairs(keys) do

        local topkKey = string.gsub(zsetKey, '^topkTempZSet:', '')

        -- If the Top-K key already exists, record it and skip
        local existing = redis.call('EXISTS', topkKey)
        if existing == 1 then
            skipped[#skipped + 1] = topkKey
        else
            local k = tonumber(redis.call('ZCARD', zsetKey)) or 0

            if k > 0 then
                local width = k * 8
                local depth = 7
                local decay = 0.9

                redis.call('TOPK.RESERVE', topkKey, k, width, depth, decay)

                local members = redis.call('ZREVRANGE', zsetKey, 0, -1, 'WITHSCORES')

                local BATCH_CAP = 500

                local j = 1
                while j <= #members - 1 do
                    local item  = unwrap(members[j])
                    local score = math.floor(tonumber(unwrap(members[j + 1])) or 0)

                    local remaining = score
                    while remaining > 0 do
                        local batch = math.min(remaining, BATCH_CAP)
                        local args  = {}
                        for _ = 1, batch do
                            args[#args + 1] = item
                        end
                        redis.call('TOPK.ADD', topkKey, unpack(args))
                        remaining = remaining - batch
                    end

                    j = j + 2
                end

                count = count + 1
            end
        end
    end

until cursor == "0"

-- Build the response string
local response = "Successfully restored " .. count .. " Top-K keys from sorted sets."

if #skipped > 0 then
    response = response .. "\\nSkipped " .. #skipped .. " key(s) that already exist: "
    for idx, name in ipairs(skipped) do
        if idx == 1 then
            response = response .. name
        else
            response = response .. ", " .. name
        end
    end
end

return response'''
print(f'Result from lua_to_json: {dfo("EVAL",lua_to_topk,'0')}')